In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd 
import numpy as np
import openpyxl
import re
import warnings
from src.config import DATA_RAW, DATA_PROCESSED
from IPython.display import display
from src.preprocessing import clean_sponsor_columns
from src.data_loader import load_raw, save_processed, load_processed
from src.preprocessing import clean_data, check_missing_values, check_duplicates, clean_sponsor_columns, clean_region_column
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:.2f}'.format

In [2]:
df_dirty = load_raw('Turan_oyun günü sorğusu.xlsx')
df = df_dirty.copy()
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(r'\s+', ' ', regex=True).str.strip()
df.head()

'Turan_oyun günü sorğusu.xlsx' Excel faylı kimi oxundu. Shape: (52, 49)


,Timestamp,Turan Tovuz PFK-nın oyunlarını hansı tezlikdə izləyirsiniz?,Hansı səbəbdən oyunları tez tez izləmirsiniz?,Oyunları əsasən necə izləyirsiniz?,Oyunları hansı səbəbdən stadionda izləmirsiniz?,Oyun barədə məlumatı haradan əldə etmisiniz?,Bileti haradan almısınız?,Stadiona getmək üçün hansı vasitədən istifadə etmisiniz?,Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?,"Stadionda son izlədiyiniz oyunu sizinlə gələnlərin arasında uşaq var idi? bir neçə uşaq gəlmişdirsə, sayı ""digər"" bölməsində qeyd edə bilərsiniz","Sizcə, oyuna ailənizlə gəlmək üçün şərait uyğundurmu?",Xahiş edirik stadiona gedərkən üzləşdiyiniz problemləri qeyd edin:,Xahiş edirik stadionda üzləşdiyiniz problemləri qeyd edin:,Xahiş edirik stadiondan çıxarkən üzləşdiyiniz problemləri qeyd edin:,Stadion ərazisində satışda olan qida və içkilər almısınızmı?,Oyun zamanı satışda olan qida və içkilərə ortalama nə qədər pul xərcləmisiniz? məbləğ(AZN),Satış məntəqələrinin əlçatan olması,Qida və içkilərin çeşidləri,Qida və içkilərin keyfiyyəti,Qida və içkilərin qiyməti,Stadion ərazisindəki fanşopdan klubun məhsullarını almısınızmı?,Oyun zamanı satışda olan klubun məhsullarına nə qədər pul xərcləmisiniz?,Satış məntəqələrinin əlçatan olması 2,Məhsulların çeşidləri,Məhsulların keyfiyyəti,Bilet alma prosesini necə qiymətləndirirsiniz?,Bilet yoxlayan əməkdaşların işini necə qiymətləndirirsiniz?,Təhlükəsizlik xidmətinin işini necə qiymətləndirirsiniz?,"Könüllülərin işini necə qiymətləndirirsiniz? (istiqamətin göstərilməsi, sualların cavablandırılması və s.)",Tribunaların təmizliyini necə qiymətləndirirsiniz?,Tualetlərin təmizliyini necə qiymətləndirirsiniz?,Stadionda olan ab-havanı necə qiymətləndirirsiniz?,Ümumi oyunun təşkilini necə qiymətləndirirsiniz?,Klubun faəliyyətini necə dəyərləndirirsiniz?,Sponsorun dəstəyini necə dəyərləndirirsiniz?,Stadionda neçə sponsor reklamı gördünüz?,Bunlar hansılardır?,"Son oyundakı təcrübənizə əsasən, Turan Tovuz PFK-nın növbəti oyunlarını stadionda izləmək istəyərdiniz?","Son oyundakı təcrübənizə əsasən, dostlarınıza Turan Tovuz PFK-nın oyunlarını stadionda izləməyi tövsiyə edərsiniz?","Ad, soyad",Cins,Yaşınız,Məşğuliyyət,Ailə vəziyyəti,Turan Tovuz PFK-nın fəaliyyəti ilə bağlı məlumat almaq istəyirsinizmi?,Qeydlər,"Harada yaşayırsınız, oyunu izləməyə haradan gəlmisiniz? rayon/kəndin adını qeyd edin",Məhsulların qiyməti,Column 46
0,2025-03-19 15:47:08.896,Hər oyununu izləyirəm,NaN,Stadionda baxıram,NaN,Sosial media kanallarından,Stadionun kassasından,Piyada,2,Xeyr,Bəli,Qarşı tribunanın üstünün bağlı olması,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Bəli,1,10.00,5.00,5.00,5.00,Xeyr,NaN,NaN,NaN,NaN,10,7,10,9,10,10.00,10,6,10,10,1,huner group,Bəli,Bəli,Tural İmanov,Kişi,37,İşləyirəm,Evli,Bəli,NaN,Tovuz şəhəri,NaN,NaN
1,2025-03-18 12:57:01.184,Hər oyununu izləyirəm,NaN,Stadionda baxıram,NaN,Şəhərdə yerləşən posterlərdən,Stadionun kassasından,Piyada,12,Xeyr,Bəli,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Bəli,1,10.00,10.00,10.00,10.00,Xeyr,NaN,NaN,NaN,NaN,10,10,10,10,10,10.00,10,10,10,10,3,hg/tovuz su/FİF,Bəli,Bəli,Rüstəm Zeynalov,Kişi,42,İşsizəm,Evli,Bəli,NaN,NaN,NaN,NaN
2,2025-03-18 12:07:38.943,Hər oyununu izləyirəm,NaN,Stadionda baxıram,NaN,Tanışlardan,Dəvətnamə veriblər,Taksi,1,Bəli,Bəli,"Stadionun parkinqində yerin az olması, Heç bir...",Tütün məhsullarının istifadəsi. Smiçka çırtdam...,"Stadiondan çıxış zamanı sıxlıq olması, Heç bir...",Bəli,1,8.00,5.00,7.00,5.00,Bəli,10,8.00,5.00,8.00,10,10,10,10,10,5.00,10,10,8,10,3,"huner group, oyal, böyük qışlaq su",Bəli,Bəli,Gülnarə Qədirova,Qadın,33,İşsizəm,Evli,Xeyr,Yaşasın Tovuz,NaN,NaN,NaN
3,2025-03-19 14:06:35.732,Hər oyununu izləyirəm,NaN,Stadionda baxıram,NaN,Sosial media kanallarından,Stadionun kassasından,İctimai nəqliyyat,8,Bəli,Bəli,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Bəli,2,1.00,8.00,10.00,3.00,Bəli,2,10.00,10.00,10.00,10,10,10,10,10,10.00,10,10,10,10,1,Huner qurup

In [3]:
df.columns = (
    df.columns
    .str.replace(r'\s+', ' ', regex=True) 
    .str.strip() 
    .str.replace(r'\s+([?.,!])', r'\1', regex=True)  
)

In [4]:
info_df = pd.DataFrame({
    'Column': [col[:30] + '...' if len(col) > 30 else col for col in df.columns],
    'Dtype': df.dtypes.values,
    'Non-Null Count': df.notnull().sum().values
})

print(info_df.to_string())

                               Column           Dtype  Non-Null Count
0                           Timestamp  datetime64[us]              52
1   Turan Tovuz PFK-nın oyunlarını...             str              52
2   Hansı səbəbdən oyunları tez te...             str               5
3   Oyunları əsasən necə izləyirsi...             str              51
4   Oyunları hansı səbəbdən stadio...             str               9
5   Oyun barədə məlumatı haradan ə...             str              52
6           Bileti haradan almısınız?             str              52
7   Stadiona getmək üçün hansı vas...             str              52
8   Stadionda son izlədiyiniz oyun...          object              52
9   Stadionda son izlədiyiniz oyun...          object              52
10  Sizcə, oyuna ailənizlə gəlmək ...             str              52
11  Xahiş edirik stadiona gedərkən...             str              52
12  Xahiş edirik stadionda üzləşdi...             str              52
13  Xahiş edirik sta

In [5]:
df.describe()

,Timestamp,Satış məntəqələrinin əlçatan olması,Qida və içkilərin çeşidləri,Qida və içkilərin keyfiyyəti,Qida və içkilərin qiyməti,Satış məntəqələrinin əlçatan olması 2,Məhsulların çeşidləri,Məhsulların keyfiyyəti,Bilet alma prosesini necə qiymətləndirirsiniz?,Bilet yoxlayan əməkdaşların işini necə qiymətləndirirsiniz?,Təhlükəsizlik xidmətinin işini necə qiymətləndirirsiniz?,"Könüllülərin işini necə qiymətləndirirsiniz? (istiqamətin göstərilməsi, sualların cavablandırılması və s.)",Tribunaların təmizliyini necə qiymətləndirirsiniz?,Tualetlərin təmizliyini necə qiymətləndirirsiniz?,Stadionda olan ab-havanı necə qiymətləndirirsiniz?,Ümumi oyunun təşkilini necə qiymətləndirirsiniz?,Klubun faəliyyətini necə dəyərləndirirsiniz?,Sponsorun dəstəyini necə dəyərləndirirsiniz?,Məhsulların qiyməti,Column 46
count,52,29.00,29.00,29.00,29.00,9.00,9.00,9.00,52.00,52.00,52.00,52.00,52.00,51.00,52.00,52.00,52.00,52.00,4.00,0.00
mean,2025-03-18 21:51:51.605365,8.00,7.59,8.03,6.34,8.89,7.67,9.11,8.98,8.69,9.06,8.58,8.50,7.92,9.44,8.42,8.33,9.35,8.25,NaN
min,2025-03-17 19:51:21.764000,1.00,3.00,1.00,1.00,5.00,3.00,6.00,3.00,3.00,5.00,1.00,1.00,1.00,3.00,3.00,3.00,3.00,5.00,NaN
25%,2025-03-18 12:52:02.381500,7.00,6.00,7.00,5.00,8.00,5.00,8.00,8.00,8.00,8.00,8.00,8.00,6.00,10.00,7.75,7.00,10.00,7.25,NaN
50%,2025-03-18 14:27:20.239000,8.00,8.00,9.00,6.00,10.00,10.00,10.00,10.00,9.50,10.00,10.00,10.00,9.00,10.00,9.00,9.00,10.00,9.00,NaN
75%,2025-03-19 11:49:31.846250,10.00,10.00,10.00,8.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,NaN
max,2025-03-24 20:58:29.613000,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,NaN
std,NaN,2.28,2.15,2.29,2.72,1.83,2.87,1.45,1.58,1.81,1.30,2.40,2.42,2.27,1.43,1.74,1.93,1.52,2.36,NaN


In [6]:
check_duplicates(df)

np.int64(0)

In [7]:
check_missing_values(df).head(17)

,missing_values
Column 46,52
Məhsulların qiyməti,48
Hansı səbəbdən oyunları tez tez izləmirsiniz?,47
Oyunları hansı səbəbdən stadionda izləmirsiniz?,43
Məhsulların keyfiyyəti,43
Məhsulların çeşidləri,43
Satış məntəqələrinin əlçatan olması 2,43
Oyun zamanı satışda olan klubun məhsullarına nə qədər pul xərcləmisiniz?,43
Qeydlər,35
Qida və içkilərin keyfiyyəti,23


In [8]:
df.columns = df.columns.str.strip()  # boşluqları at
df.columns = df.columns.str.replace('\n', ' ')  # sətir daxilində \n varsa

In [9]:
df = df.drop(columns=['Column 46'])

In [10]:
df['Stadion ərazisindəki fanşopdan klubun məhsullarını almısınızmı?'].value_counts(dropna=False)

Stadion ərazisindəki fanşopdan klubun məhsullarını almısınızmı?
Xeyr    43
Bəli     9
Name: count, dtype: int64

In [11]:
df['Məhsulların qiyməti'].value_counts(dropna=False)

Məhsulların qiyməti
NaN      48
10.00     2
8.00      1
5.00      1
Name: count, dtype: int64

In [12]:
null_columns = df.columns[df.isnull().any()].tolist()

for col in null_columns:
    print(f"Column: {col}")
    print(df[col].value_counts(dropna=False))
    print("\n")



Column: Hansı səbəbdən oyunları tez tez izləmirsiniz?
Hansı səbəbdən oyunları tez tez izləmirsiniz?
NaN                               47
İzləməyə vaxtım yoxdur             2
Futbolçular zəifdir                1
Oyunlar barədə məlumatım olmur     1
futbola marağ yoxdu                1
Name: count, dtype: int64


Column: Oyunları əsasən necə izləyirsiniz?
Oyunları əsasən necə izləyirsiniz?
Stadionda baxıram              44
Onlayn izləyirəm                4
Evdə, televizorda izləyirəm     3
NaN                             1
Name: count, dtype: int64


Column: Oyunları hansı səbəbdən stadionda izləmirsiniz?
Oyunları hansı səbəbdən stadionda izləmirsiniz?
NaN                                           43
Stadiona getməyə vaxtım olmur                  2
Stadionda izləməyi sevmirəm                    2
Stadiona getməyə vaxtım olmur, İşdə oluram     1
Komandanının zəifliyinə görə                   1
bakı da qalıram                                1
Stadion uzaq yerləşdiyinə görə,                

In [13]:
def parse_money(value):
    if pd.isna(value):
        return None  # NaN olaraq qalır (sual aid deyildi)
    
    value = str(value).strip().lower()
    
    # "forma" kimi qeyri-pul cavabları — ayrıca qeyd üçün saxla, rəqəmə çevirmə
    if not re.search(r'\d', value):
        return None
    
    # Aralıq varsa (3-4, 5-6), ortasını götür
    range_match = re.match(r'(\d+)\s*-\s*(\d+)', value)
    if range_match:
        low, high = map(int, range_match.groups())
        return (low + high) / 2
    
    # "7-dən çox" kimi aşağı sərhəd
    if 'çox' in value:
        num = re.search(r'\d+', value)
        return int(num.group()) + 1 if num else None  # təxmini
    
    # Sadə rəqəm (mətnlə qarışıq: "10 AZN", "5manat")
    num_match = re.search(r'\d+', value)
    return int(num_match.group()) if num_match else None

df['Oyun zamanı satışda olan qida və içkilərə ortalama nə qədər pul xərcləmisiniz? məbləğ(AZN)'] = df['Oyun zamanı satışda olan qida və içkilərə ortalama nə qədər pul xərcləmisiniz? məbləğ(AZN)'].apply(parse_money)
df['Oyun zamanı satışda olan klubun məhsullarına nə qədər pul xərcləmisiniz?'] = df['Oyun zamanı satışda olan klubun məhsullarına nə qədər pul xərcləmisiniz?'].apply(parse_money)

In [14]:
#Araşdırılan məlumata əsasən oyuna baxmağa gəlmişdir
df['Oyunları əsasən necə izləyirsiniz?'] = df['Oyunları əsasən necə izləyirsiniz?'].fillna('Stadionda baxıram')

In [15]:
conditional_fill = {
    'Hansı səbəbdən oyunları tez tez izləmirsiniz?': 'Qeyd olunmayıb',
    'Oyunları hansı səbəbdən stadionda izləmirsiniz?': 'Qeyd olunmayıb',
}
for col, val in conditional_fill.items():
    df[col] = df[col].fillna(val)

# 3. Könüllü mətn sualları
df['Harada yaşayırsınız, oyunu izləməyə haradan gəlmisiniz? rayon/kəndin adını qeyd edin'] = \
    df['Harada yaşayırsınız, oyunu izləməyə haradan gəlmisiniz? rayon/kəndin adını qeyd edin'].fillna('Qeyd olunmayıb')


# 4. Qeydlər — əvvəlcə boşluq-string-ləri təmizlə, sonra doldur
df['Qeydlər'] = df['Qeydlər'].replace(r'^\s*$', pd.NA, regex=True)
df['Qeydlər'] = df['Qeydlər'].fillna('Şərh yoxdur')

print("Qalan NaN sayı:\n", df.isnull().sum()[df.isnull().sum() > 0])

Qalan NaN sayı:
 Oyun zamanı satışda olan qida və içkilərə ortalama nə qədər pul xərcləmisiniz? məbləğ(AZN)    23
Satış məntəqələrinin əlçatan olması                                                           23
Qida və içkilərin çeşidləri                                                                   23
Qida və içkilərin keyfiyyəti                                                                  23
Qida və içkilərin qiyməti                                                                     23
Oyun zamanı satışda olan klubun məhsullarına nə qədər pul xərcləmisiniz?                      44
Satış məntəqələrinin əlçatan olması 2                                                         43
Məhsulların çeşidləri                                                                         43
Məhsulların keyfiyyəti                                                                        43
Tualetlərin təmizliyini necə qiymətləndirirsiniz?                                              1
Məhsulların q

In [16]:
df[df['Oyunları əsasən necə izləyirsiniz?'] == "Stadionda baxıram"].shape[0]

45

In [17]:
df['Bunlar hansılardır?'].value_counts(dropna=False)

Bunlar hansılardır?
huner group                                        11
0                                                   7
Hünər Qrup                                          5
Hünər qrup                                          3
2                                                   2
huner group                                         2
Huner Group                                         2
hg/tovuz su/FİF                                     1
huner group, oyal, böyük qışlaq su                  1
Huner qurup                                         1
Hünər Qrup Tovuz su və digərləri                    1
huner                                               1
taniks/az group/hüner group/kapital bank plakat     1
Hüner group/tovuz mall                              1
yadında deyil                                       1
Huner group, Tovuz Mall                             1
Tovuz su. Hünər qurup. Racism. Misli. Pfl. az       1
Böyük qışlaq suyu hauder                            1
huner, t

In [18]:
df = clean_sponsor_columns(df)

In [19]:
df.head()

,Timestamp,Turan Tovuz PFK-nın oyunlarını hansı tezlikdə izləyirsiniz?,Hansı səbəbdən oyunları tez tez izləmirsiniz?,Oyunları əsasən necə izləyirsiniz?,Oyunları hansı səbəbdən stadionda izləmirsiniz?,Oyun barədə məlumatı haradan əldə etmisiniz?,Bileti haradan almısınız?,Stadiona getmək üçün hansı vasitədən istifadə etmisiniz?,Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?,"Stadionda son izlədiyiniz oyunu sizinlə gələnlərin arasında uşaq var idi? bir neçə uşaq gəlmişdirsə, sayı ""digər"" bölməsində qeyd edə bilərsiniz","Sizcə, oyuna ailənizlə gəlmək üçün şərait uyğundurmu?",Xahiş edirik stadiona gedərkən üzləşdiyiniz problemləri qeyd edin:,Xahiş edirik stadionda üzləşdiyiniz problemləri qeyd edin:,Xahiş edirik stadiondan çıxarkən üzləşdiyiniz problemləri qeyd edin:,Stadion ərazisində satışda olan qida və içkilər almısınızmı?,Oyun zamanı satışda olan qida və içkilərə ortalama nə qədər pul xərcləmisiniz? məbləğ(AZN),Satış məntəqələrinin əlçatan olması,Qida və içkilərin çeşidləri,Qida və içkilərin keyfiyyəti,Qida və içkilərin qiyməti,Stadion ərazisindəki fanşopdan klubun məhsullarını almısınızmı?,Oyun zamanı satışda olan klubun məhsullarına nə qədər pul xərcləmisiniz?,Satış məntəqələrinin əlçatan olması 2,Məhsulların çeşidləri,Məhsulların keyfiyyəti,Bilet alma prosesini necə qiymətləndirirsiniz?,Bilet yoxlayan əməkdaşların işini necə qiymətləndirirsiniz?,Təhlükəsizlik xidmətinin işini necə qiymətləndirirsiniz?,"Könüllülərin işini necə qiymətləndirirsiniz? (istiqamətin göstərilməsi, sualların cavablandırılması və s.)",Tribunaların təmizliyini necə qiymətləndirirsiniz?,Tualetlərin təmizliyini necə qiymətləndirirsiniz?,Stadionda olan ab-havanı necə qiymətləndirirsiniz?,Ümumi oyunun təşkilini necə qiymətləndirirsiniz?,Klubun faəliyyətini necə dəyərləndirirsiniz?,Sponsorun dəstəyini necə dəyərləndirirsiniz?,Stadionda neçə sponsor reklamı gördünüz?,Bunlar hansılardır?,"Son oyundakı təcrübənizə əsasən, Turan Tovuz PFK-nın növbəti oyunlarını stadionda izləmək istəyərdiniz?","Son oyundakı təcrübənizə əsasən, dostlarınıza Turan Tovuz PFK-nın oyunlarını stadionda izləməyi tövsiyə edərsiniz?","Ad, soyad",Cins,Yaşınız,Məşğuliyyət,Ailə vəziyyəti,Turan Tovuz PFK-nın fəaliyyəti ilə bağlı məlumat almaq istəyirsinizmi?,Qeydlər,"Harada yaşayırsınız, oyunu izləməyə haradan gəlmisiniz? rayon/kəndin adını qeyd edin",Məhsulların qiyməti
0,2025-03-19 15:47:08.896,Hər oyununu izləyirəm,Qeyd olunmayıb,Stadionda baxıram,Qeyd olunmayıb,Sosial media kanallarından,Stadionun kassasından,Piyada,2,Xeyr,Bəli,Qarşı tribunanın üstünün bağlı olması,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Bəli,1.00,10.00,5.00,5.00,5.00,Xeyr,NaN,NaN,NaN,NaN,10,7,10,9,10,10.00,10,6,10,10,1,HUNER GROUP,Bəli,Bəli,Tural İmanov,Kişi,37,İşləyirəm,Evli,Bəli,Şərh yoxdur,Tovuz şəhəri,NaN
1,2025-03-18 12:57:01.184,Hər oyununu izləyirəm,Qeyd olunmayıb,Stadionda baxıram,Qeyd olunmayıb,Şəhərdə yerləşən posterlərdən,Stadionun kassasından,Piyada,12,Xeyr,Bəli,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Bəli,1.00,10.00,10.00,10.00,10.00,Xeyr,NaN,NaN,NaN,NaN,10,10,10,10,10,10.00,10,10,10,10,3,"HUNER GROUP, Tovuz Su, FİF",Bəli,Bəli,Rüstəm Zeynalov,Kişi,42,İşsizəm,Evli,Bəli,Şərh yoxdur,Qeyd olunmayıb,NaN
2,2025-03-18 12:07:38.943,Hər oyununu izləyirəm,Qeyd olunmayıb,Stadionda baxıram,Qeyd olunmayıb,Tanışlardan,Dəvətnamə veriblər,Taksi,1,Bəli,Bəli,"Stadionun parkinqində yerin az olması, Heç bir...",Tütün məhsullarının istifadəsi. Smiçka çırtdam...,"Stadiondan çıxış zamanı sıxlıq olması, Heç bir...",Bəli,1.00,8.00,5.00,7.00,5.00,Bəli,10.00,8.00,5.00,8.00,10,10,10,10,10,5.00,10,10,8,10,3,"HUNER GROUP, Oyal, Böyük Qışlaq Su",Bəli,Bəli,Gülnarə Qədirova,Qadın,33,İşsizəm,Evli,Xeyr,Yaşasın Tovuz,Qeyd olunmayıb,NaN
3,2025-03-19 14:06:35.732,Hər oyununu izləyirəm,Qeyd olunmayıb,Stadionda baxıram,Qeyd olunmayıb,Sosial media kanallarından,Stadionun kassasından,İctimai nəqliyyat,8,Bəli,Bəli,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim

In [20]:
df.shape


(52, 48)

In [21]:
df = clean_sponsor_columns(df)

df['Bunlar hansılardır?'].value_counts(dropna=False)

Bunlar hansılardır?
HUNER GROUP                                   29
NaN                                           11
Tovuz Su, HUNER GROUP                          3
HUNER GROUP, Tovuz Su                          2
HUNER GROUP, Tovuz Mall                        2
HUNER GROUP, Tovuz Su, FİF                     1
HUNER GROUP, Oyal, Böyük Qışlaq Su             1
HUNER GROUP, Ada Store                         1
Az Group, HUNER GROUP, Kapital Bank Plakat     1
Böyük Qışlaq Su                                1
Name: count, dtype: int64

In [22]:
df['Bunlar hansılardır?'] = df['Bunlar hansılardır?'].astype(str).str.strip().replace(['görmədim', 'yadında deyil'], np.nan)
df['Bunlar hansılardır?'] = df['Bunlar hansılardır?'].astype(str).str.strip().replace(r'^2$', 'HUNER GROUP, Ada Store', regex=True)

In [23]:
df['Bunlar hansılardır?'] = df['Bunlar hansılardır?'].astype(str).str.strip().replace(['hünər qurup', 'hüner qrup', 'hünər grup','HUNER GROUP,', 'huner', 'hünər qrup'], 'HUNER GROUP')

In [24]:
df['Bunlar hansılardır?'].value_counts(dropna=False)

Bunlar hansılardır?
HUNER GROUP                                   29
NaN                                           11
Tovuz Su, HUNER GROUP                          3
HUNER GROUP, Tovuz Su                          2
HUNER GROUP, Tovuz Mall                        2
HUNER GROUP, Tovuz Su, FİF                     1
HUNER GROUP, Oyal, Böyük Qışlaq Su             1
HUNER GROUP, Ada Store                         1
Az Group, HUNER GROUP, Kapital Bank Plakat     1
Böyük Qışlaq Su                                1
Name: count, dtype: int64

In [25]:
df['Stadionda neçə sponsor reklamı gördünüz?'].value_counts(dropna=False)


Stadionda neçə sponsor reklamı gördünüz?
1    30
0    11
2     8
3     3
Name: count, dtype: int64

In [26]:
df['Stadionda neçə sponsor reklamı gördünüz?'].apply(pd.to_numeric, errors='coerce')
df['Stadionda neçə sponsor reklamı gördünüz?'] = df['Stadionda neçə sponsor reklamı gördünüz?'].astype('Int64')

In [27]:
df.head()

,Timestamp,Turan Tovuz PFK-nın oyunlarını hansı tezlikdə izləyirsiniz?,Hansı səbəbdən oyunları tez tez izləmirsiniz?,Oyunları əsasən necə izləyirsiniz?,Oyunları hansı səbəbdən stadionda izləmirsiniz?,Oyun barədə məlumatı haradan əldə etmisiniz?,Bileti haradan almısınız?,Stadiona getmək üçün hansı vasitədən istifadə etmisiniz?,Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?,"Stadionda son izlədiyiniz oyunu sizinlə gələnlərin arasında uşaq var idi? bir neçə uşaq gəlmişdirsə, sayı ""digər"" bölməsində qeyd edə bilərsiniz","Sizcə, oyuna ailənizlə gəlmək üçün şərait uyğundurmu?",Xahiş edirik stadiona gedərkən üzləşdiyiniz problemləri qeyd edin:,Xahiş edirik stadionda üzləşdiyiniz problemləri qeyd edin:,Xahiş edirik stadiondan çıxarkən üzləşdiyiniz problemləri qeyd edin:,Stadion ərazisində satışda olan qida və içkilər almısınızmı?,Oyun zamanı satışda olan qida və içkilərə ortalama nə qədər pul xərcləmisiniz? məbləğ(AZN),Satış məntəqələrinin əlçatan olması,Qida və içkilərin çeşidləri,Qida və içkilərin keyfiyyəti,Qida və içkilərin qiyməti,Stadion ərazisindəki fanşopdan klubun məhsullarını almısınızmı?,Oyun zamanı satışda olan klubun məhsullarına nə qədər pul xərcləmisiniz?,Satış məntəqələrinin əlçatan olması 2,Məhsulların çeşidləri,Məhsulların keyfiyyəti,Bilet alma prosesini necə qiymətləndirirsiniz?,Bilet yoxlayan əməkdaşların işini necə qiymətləndirirsiniz?,Təhlükəsizlik xidmətinin işini necə qiymətləndirirsiniz?,"Könüllülərin işini necə qiymətləndirirsiniz? (istiqamətin göstərilməsi, sualların cavablandırılması və s.)",Tribunaların təmizliyini necə qiymətləndirirsiniz?,Tualetlərin təmizliyini necə qiymətləndirirsiniz?,Stadionda olan ab-havanı necə qiymətləndirirsiniz?,Ümumi oyunun təşkilini necə qiymətləndirirsiniz?,Klubun faəliyyətini necə dəyərləndirirsiniz?,Sponsorun dəstəyini necə dəyərləndirirsiniz?,Stadionda neçə sponsor reklamı gördünüz?,Bunlar hansılardır?,"Son oyundakı təcrübənizə əsasən, Turan Tovuz PFK-nın növbəti oyunlarını stadionda izləmək istəyərdiniz?","Son oyundakı təcrübənizə əsasən, dostlarınıza Turan Tovuz PFK-nın oyunlarını stadionda izləməyi tövsiyə edərsiniz?","Ad, soyad",Cins,Yaşınız,Məşğuliyyət,Ailə vəziyyəti,Turan Tovuz PFK-nın fəaliyyəti ilə bağlı məlumat almaq istəyirsinizmi?,Qeydlər,"Harada yaşayırsınız, oyunu izləməyə haradan gəlmisiniz? rayon/kəndin adını qeyd edin",Məhsulların qiyməti
0,2025-03-19 15:47:08.896,Hər oyununu izləyirəm,Qeyd olunmayıb,Stadionda baxıram,Qeyd olunmayıb,Sosial media kanallarından,Stadionun kassasından,Piyada,2,Xeyr,Bəli,Qarşı tribunanın üstünün bağlı olması,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Bəli,1.00,10.00,5.00,5.00,5.00,Xeyr,NaN,NaN,NaN,NaN,10,7,10,9,10,10.00,10,6,10,10,1,HUNER GROUP,Bəli,Bəli,Tural İmanov,Kişi,37,İşləyirəm,Evli,Bəli,Şərh yoxdur,Tovuz şəhəri,NaN
1,2025-03-18 12:57:01.184,Hər oyununu izləyirəm,Qeyd olunmayıb,Stadionda baxıram,Qeyd olunmayıb,Şəhərdə yerləşən posterlərdən,Stadionun kassasından,Piyada,12,Xeyr,Bəli,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Bəli,1.00,10.00,10.00,10.00,10.00,Xeyr,NaN,NaN,NaN,NaN,10,10,10,10,10,10.00,10,10,10,10,3,"HUNER GROUP, Tovuz Su, FİF",Bəli,Bəli,Rüstəm Zeynalov,Kişi,42,İşsizəm,Evli,Bəli,Şərh yoxdur,Qeyd olunmayıb,NaN
2,2025-03-18 12:07:38.943,Hər oyununu izləyirəm,Qeyd olunmayıb,Stadionda baxıram,Qeyd olunmayıb,Tanışlardan,Dəvətnamə veriblər,Taksi,1,Bəli,Bəli,"Stadionun parkinqində yerin az olması, Heç bir...",Tütün məhsullarının istifadəsi. Smiçka çırtdam...,"Stadiondan çıxış zamanı sıxlıq olması, Heç bir...",Bəli,1.00,8.00,5.00,7.00,5.00,Bəli,10.00,8.00,5.00,8.00,10,10,10,10,10,5.00,10,10,8,10,3,"HUNER GROUP, Oyal, Böyük Qışlaq Su",Bəli,Bəli,Gülnarə Qədirova,Qadın,33,İşsizəm,Evli,Xeyr,Yaşasın Tovuz,Qeyd olunmayıb,NaN
3,2025-03-19 14:06:35.732,Hər oyununu izləyirəm,Qeyd olunmayıb,Stadionda baxıram,Qeyd olunmayıb,Sosial media kanallarından,Stadionun kassasından,İctimai nəqliyyat,8,Bəli,Bəli,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim

---------

In [28]:
# Burada 40 yazmışam, çünki 1985 martın 14 tarixində doğulmuş şəxs 2025-ci ildə 40 yaşındadır
# və 34 yazmağımın səbəbi isə Hüseynov Əsgər adlı şəxsin digər məlumat datasına əsasən 34 yaşında olmasıdır
df['Yaşınız'] = df['Yaşınız'].astype(str).str.strip().replace('1985 martın 14', 40)
df['Yaşınız'] = df['Yaşınız'].astype(str).str.strip().replace('Tovuz', 34)

In [29]:
df['Yaşınız'] = pd.to_numeric(df['Yaşınız'], errors='coerce').astype('Int64')

---------

In [30]:
df['Stadionda son izlədiyiniz oyunu sizinlə gələnlərin arasında uşaq var idi? bir neçə uşaq gəlmişdirsə, sayı "digər" bölməsində qeyd edə bilərsiniz'] = \
    df['Stadionda son izlədiyiniz oyunu sizinlə gələnlərin arasında uşaq var idi? bir neçə uşaq gəlmişdirsə, sayı "digər" bölməsində qeyd edə bilərsiniz'].astype(str).str.strip().replace(['Dostunun qızı ilə', '3 / 1 oğ 2 qiz', '2'], 'Bəli')

-----


In [31]:
df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'] = df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'].astype(str).str.strip().replace(['3-4'], 3)
df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'] = df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'].astype(str).str.strip().replace(['Tək gəlmişəm'], 0)
df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'] = df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'].astype(str).str.strip().replace(['10+'], 10)
df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'] = df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'].astype(str).str.strip().replace(['2 Oğul ilə gəlib'], 2)
df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'] = df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'].astype(str).str.strip().replace(['Dostları ilə'], 3)

In [32]:
df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'] = df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'].apply(pd.to_numeric, errors='coerce')
df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'] = df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'].astype('Int64')

-------

In [33]:
df['Sizcə, oyuna ailənizlə gəlmək üçün şərait uyğundurmu?'] = df['Sizcə, oyuna ailənizlə gəlmək üçün şərait uyğundurmu?'].astype(str).str.strip().replace(['50 faiz', 'Ailəli yer olsun yaxşı olar', 'Ancaq uşaqlarla gəlməyə uygundur', 'Bəzi yerlərdə bəli vip', 'Xüsusi zona olsa gəlmək olar'], 'Qismən').replace(['Söyüş söyülməsi', 'çox bərbaddı. natəmizlik , söüş, semiçka', 'Söyüş çox söyülür ailə üçün xüsusi yer yoxdu', 'Söüşün söyülməsi', 'Ailə ilə gəlmək olmur söyüş söyülür.'], 'Xeyr').replace(['he'], 'Bəli')

In [34]:
df['Sizcə, oyuna ailənizlə gəlmək üçün şərait uyğundurmu?'].value_counts(dropna=False)

Sizcə, oyuna ailənizlə gəlmək üçün şərait uyğundurmu?
Bəli      26
Xeyr      21
Qismən     5
Name: count, dtype: int64

In [35]:
df['Oyun zamanı satışda olan qida və içkilərə ortalama nə qədər pul xərcləmisiniz? məbləğ(AZN)'].value_counts

<bound method IndexOpsMixin.value_counts of 0     1.00
1     1.00
2     1.00
3     2.00
4     2.00
5     2.00
6     4.00
7     5.00
8     5.00
9     5.00
10    6.00
11    6.00
12    7.00
13   10.00
14   10.00
15   10.00
16   15.00
17   18.00
18   20.00
19   20.00
20   10.00
21   10.00
22    3.50
23    5.50
24    5.50
25    5.00
26    7.50
27    8.00
28    8.00
29     NaN
30     NaN
31     NaN
32     NaN
33     NaN
34     NaN
35     NaN
36     NaN
37     NaN
38     NaN
39     NaN
40     NaN
41     NaN
42     NaN
43     NaN
44     NaN
45     NaN
46     NaN
47     NaN
48     NaN
49     NaN
50     NaN
51     NaN
Name: Oyun zamanı satışda olan qida və içkilərə ortalama nə qədər pul xərcləmisiniz? məbləğ(AZN), dtype: float64>

In [36]:
df['Harada yaşayırsınız, oyunu izləməyə haradan gəlmisiniz? rayon/kəndin adını qeyd edin'].value_counts(dropna=False)

Harada yaşayırsınız, oyunu izləməyə haradan gəlmisiniz? rayon/kəndin adını qeyd edin
Qeyd olunmayıb                  15
Şəhərin özündə                   4
Tovuz r                          2
Tovuz şəhəri                     1
Tovuz R.N Azafli kend            1
Düz cırdaxan kəndi               1
Əlimərdanlı                      1
Cəlilli                          1
Qovlar                           1
Köhnə qala - Tovuzun kəndi       1
tovuz r                          1
Alakol                           1
tovuz rayonu                     1
Tovuz Şəhərinin özündən          1
Tovuz r_n duz cirdaxan kəndi     1
Aşağı quşçu                      1
Aşğıqulşu                        1
Bakı                             1
bozalqanlı                       1
bozaqanlı                        1
əsrik çırdaxan                   1
Əsrik kəndi                      1
Əyyublu kəndində qalır           1
Kənd                             1
Qovullar                         1
Qozaqanlı                        1
Tovuz

In [37]:
df['Harada yaşayırsınız, oyunu izləməyə haradan gəlmisiniz? rayon/kəndin adını qeyd edin'] = df['Harada yaşayırsınız, oyunu izləməyə haradan gəlmisiniz? rayon/kəndin adını qeyd edin'].astype(str).str.strip().replace(['Əyyublu kəndində qalır'], 'Əyyublu kəndi')
df['Harada yaşayırsınız, oyunu izləməyə haradan gəlmisiniz? rayon/kəndin adını qeyd edin'] = df['Harada yaşayırsınız, oyunu izləməyə haradan gəlmisiniz? rayon/kəndin adını qeyd edin'].astype(str).str.strip().replace(['Kənd'], 'Aşağı Quşçu kəndi')

In [38]:
df = clean_region_column(df)
df.head(2)

Diqqət: aşağıdakı dəyərlər heç bir kənd/şəhər şablonuna uyğun gəlmədi və olduğu kimi saxlanıldı (əl ilə yoxlayın):
  - Qeyd olunmayıb
  - Əyyublu kəndi


,Timestamp,Turan Tovuz PFK-nın oyunlarını hansı tezlikdə izləyirsiniz?,Hansı səbəbdən oyunları tez tez izləmirsiniz?,Oyunları əsasən necə izləyirsiniz?,Oyunları hansı səbəbdən stadionda izləmirsiniz?,Oyun barədə məlumatı haradan əldə etmisiniz?,Bileti haradan almısınız?,Stadiona getmək üçün hansı vasitədən istifadə etmisiniz?,Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?,"Stadionda son izlədiyiniz oyunu sizinlə gələnlərin arasında uşaq var idi? bir neçə uşaq gəlmişdirsə, sayı ""digər"" bölməsində qeyd edə bilərsiniz","Sizcə, oyuna ailənizlə gəlmək üçün şərait uyğundurmu?",Xahiş edirik stadiona gedərkən üzləşdiyiniz problemləri qeyd edin:,Xahiş edirik stadionda üzləşdiyiniz problemləri qeyd edin:,Xahiş edirik stadiondan çıxarkən üzləşdiyiniz problemləri qeyd edin:,Stadion ərazisində satışda olan qida və içkilər almısınızmı?,Oyun zamanı satışda olan qida və içkilərə ortalama nə qədər pul xərcləmisiniz? məbləğ(AZN),Satış məntəqələrinin əlçatan olması,Qida və içkilərin çeşidləri,Qida və içkilərin keyfiyyəti,Qida və içkilərin qiyməti,Stadion ərazisindəki fanşopdan klubun məhsullarını almısınızmı?,Oyun zamanı satışda olan klubun məhsullarına nə qədər pul xərcləmisiniz?,Satış məntəqələrinin əlçatan olması 2,Məhsulların çeşidləri,Məhsulların keyfiyyəti,Bilet alma prosesini necə qiymətləndirirsiniz?,Bilet yoxlayan əməkdaşların işini necə qiymətləndirirsiniz?,Təhlükəsizlik xidmətinin işini necə qiymətləndirirsiniz?,"Könüllülərin işini necə qiymətləndirirsiniz? (istiqamətin göstərilməsi, sualların cavablandırılması və s.)",Tribunaların təmizliyini necə qiymətləndirirsiniz?,Tualetlərin təmizliyini necə qiymətləndirirsiniz?,Stadionda olan ab-havanı necə qiymətləndirirsiniz?,Ümumi oyunun təşkilini necə qiymətləndirirsiniz?,Klubun faəliyyətini necə dəyərləndirirsiniz?,Sponsorun dəstəyini necə dəyərləndirirsiniz?,Stadionda neçə sponsor reklamı gördünüz?,Bunlar hansılardır?,"Son oyundakı təcrübənizə əsasən, Turan Tovuz PFK-nın növbəti oyunlarını stadionda izləmək istəyərdiniz?","Son oyundakı təcrübənizə əsasən, dostlarınıza Turan Tovuz PFK-nın oyunlarını stadionda izləməyi tövsiyə edərsiniz?","Ad, soyad",Cins,Yaşınız,Məşğuliyyət,Ailə vəziyyəti,Turan Tovuz PFK-nın fəaliyyəti ilə bağlı məlumat almaq istəyirsinizmi?,Qeydlər,"Harada yaşayırsınız, oyunu izləməyə haradan gəlmisiniz? rayon/kəndin adını qeyd edin",Məhsulların qiyməti
0,2025-03-19 15:47:08.896,Hər oyununu izləyirəm,Qeyd olunmayıb,Stadionda baxıram,Qeyd olunmayıb,Sosial media kanallarından,Stadionun kassasından,Piyada,2,Xeyr,Bəli,Qarşı tribunanın üstünün bağlı olması,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Bəli,1.00,10.00,5.00,5.00,5.00,Xeyr,NaN,NaN,NaN,NaN,10,7,10,9,10,10.00,10,6,10,10,1,HUNER GROUP,Bəli,Bəli,Tural İmanov,Kişi,37,İşləyirəm,Evli,Bəli,Şərh yoxdur,Tovuz şəhəri,NaN
1,2025-03-18 12:57:01.184,Hər oyununu izləyirəm,Qeyd olunmayıb,Stadionda baxıram,Qeyd olunmayıb,Şəhərdə yerləşən posterlərdən,Stadionun kassasından,Piyada,12,Xeyr,Bəli,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Bəli,1.00,10.00,10.00,10.00,10.00,Xeyr,NaN,NaN,NaN,NaN,10,10,10,10,10,10.00,10,10,10,10,3,"HUNER GROUP, Tovuz Su, FİF",Bəli,Bəli,Rüstəm Zeynalov,Kişi,42,İşsizəm,Evli,Bəli,Şərh yoxdur,Qeyd olunmayıb,NaN


In [39]:
null_columns = df.columns[df.isnull().any()].tolist()

for col in df.columns:
    print(f"Column: {col}")
    print(df[col].dtype)
    print(df[col].value_counts(dropna=False))
    print("\n")

Column: Timestamp
datetime64[us]
Timestamp
2025-03-19 15:47:08.896    1
2025-03-18 12:57:01.184    1
2025-03-18 12:07:38.943    1
2025-03-19 14:06:35.732    1
2025-03-17 20:44:32.298    1
2025-03-18 12:46:55.111    1
2025-03-18 14:54:26.549    1
2025-03-18 13:46:38.504    1
2025-03-19 15:32:44.205    1
2025-03-17 20:02:28.512    1
2025-03-18 14:06:11.826    1
2025-03-19 11:55:36.203    1
2025-03-18 14:39:40.470    1
2025-03-18 13:55:34.562    1
2025-03-18 16:28:49.937    1
2025-03-18 12:33:41.752    1
2025-03-18 12:13:15.059    1
2025-03-19 12:20:37.727    1
2025-03-18 16:46:29.588    1
2025-03-18 12:16:27.531    1
2025-03-18 13:35:45.410    1
2025-03-18 12:53:44.805    1
2025-03-17 19:51:21.764    1
2025-03-18 14:50:58.885    1
2025-03-19 21:23:00.569    1
2025-03-24 20:58:29.613    1
2025-03-17 20:21:52.390    1
2025-03-18 12:41:43.087    1
2025-03-18 12:14:12.843    1
2025-03-19 11:31:33.868    1
2025-03-19 12:10:48.106    1
2025-03-18 17:14:43.878    1
2025-03-19 15:49:28.734    1


In [40]:
df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'].apply(pd.to_numeric, errors='coerce')
df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'] = df['Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?'].astype('Int64')

In [41]:
df.head()

,Timestamp,Turan Tovuz PFK-nın oyunlarını hansı tezlikdə izləyirsiniz?,Hansı səbəbdən oyunları tez tez izləmirsiniz?,Oyunları əsasən necə izləyirsiniz?,Oyunları hansı səbəbdən stadionda izləmirsiniz?,Oyun barədə məlumatı haradan əldə etmisiniz?,Bileti haradan almısınız?,Stadiona getmək üçün hansı vasitədən istifadə etmisiniz?,Stadionda son izlədiyiniz oyunu sizinlə neçə nəfər gəlib?,"Stadionda son izlədiyiniz oyunu sizinlə gələnlərin arasında uşaq var idi? bir neçə uşaq gəlmişdirsə, sayı ""digər"" bölməsində qeyd edə bilərsiniz","Sizcə, oyuna ailənizlə gəlmək üçün şərait uyğundurmu?",Xahiş edirik stadiona gedərkən üzləşdiyiniz problemləri qeyd edin:,Xahiş edirik stadionda üzləşdiyiniz problemləri qeyd edin:,Xahiş edirik stadiondan çıxarkən üzləşdiyiniz problemləri qeyd edin:,Stadion ərazisində satışda olan qida və içkilər almısınızmı?,Oyun zamanı satışda olan qida və içkilərə ortalama nə qədər pul xərcləmisiniz? məbləğ(AZN),Satış məntəqələrinin əlçatan olması,Qida və içkilərin çeşidləri,Qida və içkilərin keyfiyyəti,Qida və içkilərin qiyməti,Stadion ərazisindəki fanşopdan klubun məhsullarını almısınızmı?,Oyun zamanı satışda olan klubun məhsullarına nə qədər pul xərcləmisiniz?,Satış məntəqələrinin əlçatan olması 2,Məhsulların çeşidləri,Məhsulların keyfiyyəti,Bilet alma prosesini necə qiymətləndirirsiniz?,Bilet yoxlayan əməkdaşların işini necə qiymətləndirirsiniz?,Təhlükəsizlik xidmətinin işini necə qiymətləndirirsiniz?,"Könüllülərin işini necə qiymətləndirirsiniz? (istiqamətin göstərilməsi, sualların cavablandırılması və s.)",Tribunaların təmizliyini necə qiymətləndirirsiniz?,Tualetlərin təmizliyini necə qiymətləndirirsiniz?,Stadionda olan ab-havanı necə qiymətləndirirsiniz?,Ümumi oyunun təşkilini necə qiymətləndirirsiniz?,Klubun faəliyyətini necə dəyərləndirirsiniz?,Sponsorun dəstəyini necə dəyərləndirirsiniz?,Stadionda neçə sponsor reklamı gördünüz?,Bunlar hansılardır?,"Son oyundakı təcrübənizə əsasən, Turan Tovuz PFK-nın növbəti oyunlarını stadionda izləmək istəyərdiniz?","Son oyundakı təcrübənizə əsasən, dostlarınıza Turan Tovuz PFK-nın oyunlarını stadionda izləməyi tövsiyə edərsiniz?","Ad, soyad",Cins,Yaşınız,Məşğuliyyət,Ailə vəziyyəti,Turan Tovuz PFK-nın fəaliyyəti ilə bağlı məlumat almaq istəyirsinizmi?,Qeydlər,"Harada yaşayırsınız, oyunu izləməyə haradan gəlmisiniz? rayon/kəndin adını qeyd edin",Məhsulların qiyməti
0,2025-03-19 15:47:08.896,Hər oyununu izləyirəm,Qeyd olunmayıb,Stadionda baxıram,Qeyd olunmayıb,Sosial media kanallarından,Stadionun kassasından,Piyada,2,Xeyr,Bəli,Qarşı tribunanın üstünün bağlı olması,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Bəli,1.00,10.00,5.00,5.00,5.00,Xeyr,NaN,NaN,NaN,NaN,10,7,10,9,10,10.00,10,6,10,10,1,HUNER GROUP,Bəli,Bəli,Tural İmanov,Kişi,37,İşləyirəm,Evli,Bəli,Şərh yoxdur,Tovuz şəhəri,NaN
1,2025-03-18 12:57:01.184,Hər oyununu izləyirəm,Qeyd olunmayıb,Stadionda baxıram,Qeyd olunmayıb,Şəhərdə yerləşən posterlərdən,Stadionun kassasından,Piyada,12,Xeyr,Bəli,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim,Bəli,1.00,10.00,10.00,10.00,10.00,Xeyr,NaN,NaN,NaN,NaN,10,10,10,10,10,10.00,10,10,10,10,3,"HUNER GROUP, Tovuz Su, FİF",Bəli,Bəli,Rüstəm Zeynalov,Kişi,42,İşsizəm,Evli,Bəli,Şərh yoxdur,Qeyd olunmayıb,NaN
2,2025-03-18 12:07:38.943,Hər oyununu izləyirəm,Qeyd olunmayıb,Stadionda baxıram,Qeyd olunmayıb,Tanışlardan,Dəvətnamə veriblər,Taksi,1,Bəli,Bəli,"Stadionun parkinqində yerin az olması, Heç bir...",Tütün məhsullarının istifadəsi. Smiçka çırtdam...,"Stadiondan çıxış zamanı sıxlıq olması, Heç bir...",Bəli,1.00,8.00,5.00,7.00,5.00,Bəli,10.00,8.00,5.00,8.00,10,10,10,10,10,5.00,10,10,8,10,3,"HUNER GROUP, Oyal, Böyük Qışlaq Su",Bəli,Bəli,Gülnarə Qədirova,Qadın,33,İşsizəm,Evli,Xeyr,Yaşasın Tovuz,Qeyd olunmayıb,NaN
3,2025-03-19 14:06:35.732,Hər oyununu izləyirəm,Qeyd olunmayıb,Stadionda baxıram,Qeyd olunmayıb,Sosial media kanallarından,Stadionun kassasından,İctimai nəqliyyat,8,Bəli,Bəli,Heç bir problemlə üzləşmədim,Heç bir problemlə üzləşmədim

In [42]:
df.shape

(52, 48)

In [43]:
save_processed(df, 'cleaned_survey_data.csv')

Fayl saxlanıldı: C:\Users\rashid\Desktop\fif_project\data\processed\cleaned_survey_data.csv (shape: (52, 48))
